In [10]:
import os 
from langchain_core.prompts import ChatMessagePromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough,RunnableBranch,Runnable,RunnableParallel
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_core.tools import tool
import re
from typing import List,Dict,Union,Optional,Any

In [2]:
import numpy as np
import pandas as pd
import matplotlib
import seaborn
import langchain
import sklearn
import glob


In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [11]:
@tool
def list_csv_files() -> Optional[List[str]]:
    """
    List all CSV file names in the local directory
    
    Returns:
    A list containing CSV file name 
    if no csv files are found, return None.
    """
    csv_files=glob.glob(os.path.join(os.getcwd(),"*.csv"))
    if not csv_files:
        return None
    return [os.path.basename(file) for file in csv_files]

In [7]:
print("Tool Name:", list_csv_files.name)
print("Tool Description:", list_csv_files.description)
print("Tool Arguments:", list_csv_files.args)

Tool Name: list_csv_files
Tool Description: List all CSV file names in the local directory

Returns:
A list containing CSV file name 
if no csv files are found, return None.
Tool Arguments: {}


#### Dataset Caching Tool

In [8]:
DATAFRAME_CACHE={}

@tool
def preload_datasets(paths: List[str]) -> str:
    """
    Loads CSV files into a global cache if not already loaded.
    
    This functions helps to efficiently manage datasets by loading them once
    and storing them in memory for future use. without caching, you would waste
    tokens describing datasets content repeatedly in agent response
    
    Args:
        paths: A list of file path to csv files.
    Returns:
        A message summarizing which datasets were loaded or already cached
    """
    
    loaded=[]
    cached=[]
    
    for path in paths:
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] =pd.read_csv(path)
            loaded.append(path)
        else:
            cached.append(path)
    return(
        f'Loaded datasets: {loaded}\n'
        f'Already cached:{cached}'
    )
            

#### Summarization tool

In [12]:
@tool
def get_dataset_summaries(dataset_paths: List[str]) -> List[Dict[str,Any]]:
    """
    Analyze multiple CSV files and return metadata summaries for each.
    
    Args:
        dataset_paths(List[str]):
            A list of file paths to CSV datasets
    Returns:
        List[Dict[str,Any]]:
            A list of summaries, one per dataset, each containing:
                -"file_name": The path of the dataset file.
                -"column_name": A list of column names in the dataset.
                -"data_types": A dictionary mapping columns names to their data
                                types(as string).
    """
    summaries=[]
    
    for path in dataset_paths:
        # Load and cache the datasets if not already cached
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] =pd.read_csv(path)
        
        df=DATAFRAME_CACHE[path]
        
        # Build summary
        summary={
            'file_name':path,
            'column_names':df.columns.tolist(),
            'data_types':df.dtypes.astype(str).to_dict()
        }
        summary.append(summary)
    return summaries

#### DataFrame method execution tool

In [ ]:
@tool
def call_dataframe_method(file_name: str, method:str) -> str:
    """
    Execute a method on a DataFrame and return the result.
    This tool lets you run simple DataFrame methods like 'head', 'tail', or 'describe'.capitalize
    on a datasets that has already been loaded and cached using 'preload_datasets'
    
    Args:
        file_name(str): The path or name of the datasets in the global cache.
        method(str): the name of the method to call on the Dataframe. Only no-argument
        method are supported (e.g. 'head','describe','info').
    
    
    """
    